# Approach 2 — Two-Stage Retrieve-and-Rerank
### Multimodal Fashion & Context Retrieval (Glance ML Assignment)

**The idea.** A CLIP bi-encoder scores image and text *independently* (one vector each),
so context ("in a modern office") and attribute binding leak. We fix precision with a
**two-stage** pipeline:

1. **Stage 1 — Recall (cheap, scalable):** FashionCLIP + ANN search in **ChromaDB** → top-N candidates.
2. **Stage 2 — Precision (expensive, bounded):** re-score the N candidates with a
   **BLIP Image-Text-Matching cross-encoder** that jointly attends over image patches and query
   tokens, then fuse: `final = α·clip + (1-α)·blip_itm`.

The re-ranker only ever sees N (~50) candidates regardless of database size, so precision
improves **without** breaking scalability to millions of images.

Two workflows: **Part A — Indexer**, **Part B — Retriever**, then evaluation.

## 0. Setup

In [ ]:
# !pip install -r requirements.txt   # uncomment on first run
import sys, os
sys.path.append(os.path.abspath("."))
from src import config
print("Device:", config.get_device())
print("Backbone:", config.BACKBONE_MODEL, "| Reranker:", config.RERANKER_MODEL)
print("Stage-1 top-N:", config.STAGE1_TOPN, "| fusion alpha:", config.RERANK_ALPHA)

## Part A — The Indexer
Encode all 1,000 images with FashionCLIP and store them in ChromaDB (the ANN recall index).
Stage 2 needs no index — it reads candidate images on demand.

In [ ]:
from src.indexer import build_index
index = build_index(force=False)
print("Indexed:", len(index["ids"]), "images | dim:", index["embeddings"].shape[1])

## Part B — The Retriever
### B.1 — Single query, two-stage

In [ ]:
from src.retriever import TwoStageRetriever
from src.evaluate import show_results
retriever = TwoStageRetriever()   # loads BLIP reranker on first use

q = "A red tie and a white shirt in a formal setting."
res = retriever.search(q, k=5, rerank=True)
for r in res:
    print(f"{r.score:.3f}  clip={r.clip_score:.2f} itm={r.rerank_score:.2f} "
          f"(was #{r.stage1_rank} pre-rerank)  {r.filename}")
show_results(q, res);

### B.2 — Does reranking actually help? (before / after)
Compare stage-1-only (CLIP recall) against the full two-stage result. The `stage1_rank`
column above already shows how far each final result moved after reranking.

In [ ]:
from src.evaluate import compare_stage1_vs_reranked
cmp = compare_stage1_vs_reranked(retriever, q, k=5)
print("Stage-1 only (CLIP):")
for r in cmp["stage1_only"]:
    print(f"   {r.clip_score:.3f}  {r.filename}")
print("\nReranked (two-stage):")
for r in cmp["reranked"]:
    print(f"   {r.score:.3f}  {r.filename}")
show_results(q + "  [CLIP only]", cmp["stage1_only"]);
show_results(q + "  [reranked]", cmp["reranked"]);

## Evaluation — the 5 assessment queries

In [ ]:
from src.evaluate import run_all
all_results = run_all(retriever, k=5, save=True)
for qq, res in all_results.items():
    print("\n=>", qq)
    for r in res:
        print(f"   {r.score:.3f}  clip={r.clip_score:.2f} itm={r.rerank_score:.2f}  {r.filename}")

## Why this beats vanilla CLIP (summary)
- **Precision** — BLIP-ITM's joint cross-attention resolves context and binding that a
  bi-encoder blurs.
- **Scalable** — stage 1 is sub-linear ANN over the full DB; stage 2 is O(N) with N≈50,
  independent of DB size. Works the same at 1M images.
- **Tunable** — `α` (`config.RERANK_ALPHA`) trades recall-similarity vs. cross-encoder
  precision; `top_n` trades cost vs. recall ceiling.
- **Zero-shot** — both models are pretrained; no task-specific training.